In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
from pathlib import Path
import sys

# ─── Third-Party Libraries ──────────────────────────────────────────────────────
from shapely.geometry import box
import pyrosm

# ─── Local Application / Shared Code ────────────────────────────────────────────
sys.path.append(str(Path().resolve().parent / 'shared_code'))
import geo_util as gu

!curl -s -H 'Cache-Control: no-cache, no-store' https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py > jg_folium.py
import jg_folium as jgf

In [ ]:
images_path = Path(r'C:\Users\joaqu\Dropbox\Apps\Overleaf\Analytics for Good\images') / 'Roads'
images_path.mkdir(parents=True, exist_ok=True)

In [ ]:
notebook_name = 'NL_roads.ipynb'
country = 'Netherlands'

In [ ]:
gu.get_remote_osm_pbf_timestamp(country)

In [ ]:
# global definitions
_data_path = Path( '../shared_data/' )

In [ ]:
logger = gu.get_logger(notebook_name,_data_path / 'logs')

In [ ]:
# download if needed
filepath = pyrosm.get_data("Netherlands", directory=_data_path)

In [ ]:
# Load the .pbf file
osm = pyrosm.OSM(filepath)

In [ ]:
nodes, edges = gu.load_or_acquire(
    _data_path,
    'nl_nodes_and_edges_all_modalities',
    osm.get_network,
    logger=logger,
    verbose_args=True,
    nodes=True, 
    network_type='all'
)

In [ ]:
edges.dtypes[edges.dtypes == 'category']


In [ ]:
edges.maxspeed.value_counts(dropna=False)

In [ ]:
edges[edges.maxspeed=='110'].explore()

In [ ]:
edges.memory_usage(deep=True).sort_values(ascending=False)


In [ ]:
def convertRepeatedStringsToCategorical(df, maxUniques=100, minRepetition=0.01):
    """
    Converts columns with string dtype to categorical if they have low cardinality.
    
    Args:
        df: Pandas or GeoPandas DataFrame.
        maxUniques: Maximum unique values to allow before skipping.
        minRepetition: Minimum average repetition per unique to justify conversion.
    
    Returns:
        A copy of the DataFrame with suitable columns converted to categorical.
    """
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        numUniques = df[col].nunique(dropna=False)
        numTotal = len(df[col])
        if numUniques <= maxUniques and (numTotal / numUniques) >= (1 / minRepetition):
            df[col] = df[col].astype('category')
    return df


In [ ]:
edges = convertRepeatedStringsToCategorical(edges)

In [ ]:
nodes.shape,edges.shape

In [ ]:
edges['highway'].value_counts(dropna=False)


In [ ]:
edges.osm_type.value_counts()

In [ ]:
print(nodes.crs)  # Check the CRS
print(edges.crs)  # Check the CRS

In [ ]:
f'{notebook_name} loaded with {nodes.shape[0]:,} nodes and {edges.shape[0]:,} edges.'

In [ ]:
gu.plotFastHistogram(
    edges['length'].dropna().to_numpy(),
    title='edge lengths'
)

In [ ]:
lon = nodes.geometry.x.mean()
lat = nodes.geometry.y.mean()

In [ ]:
latDelta = 0.018
lonDelta = 0.029
bbox = (lon - lonDelta, lat - latDelta, lon + lonDelta, lat + latDelta)

In [ ]:
bboxPolygon = box(*bbox)

In [ ]:
# Keep only nodes inside bbox
nodesSubset = nodes[nodes.geometry.within(bboxPolygon)]

# Keep edges that intersect the bbox (even if one node is outside)
edgesSubset = edges[edges.geometry.intersects(bboxPolygon)]

In [ ]:
nodesSubset.shape, edgesSubset.shape

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from folium import LayerControl

# Create a GeoDataFrame from the bboxPolygon
bboxGDF = gpd.GeoDataFrame(geometry=[bboxPolygon], crs='EPSG:4326')

# First: edges in blue
m = edgesSubset.explore(
    color='blue',
    style_kwds={'weight': 5},  # This sets the linewidth
    name='Edges',
    tooltip=False
)

# Second: nodes in red
nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 3},
    name='Nodes'
)

# Third: bounding box in black (transparent fill)
bboxGDF.explore(
    m=m,
    name='Bounding Box',
    color='black',
    fill=False,
    style_kwds={'weight': 8}  # This sets the linewidth
)

# Optional: add layer control
LayerControl(collapsed=False).add_to(m)
jgf.FoliumToPng(m, str( images_path / 'detail_center_box' ) )
m


In [ ]:
# Keep only rows where 'highway' is not null
edgesVisible = edgesSubset[edgesSubset['highway'].notna()].copy()

# Ensure 'highway' is a category *only with the present values*
edgesVisible['highway'] = edgesVisible['highway'].astype('category')
edgesVisible['highway'] = edgesVisible['highway'].cat.remove_unused_categories()

m = edgesVisible.explore(
    column='highway',
    categorical=True,
    legend=True,
    cmap='tab20',
    style_kwds={'weight': 5},
    name='Edges by highway',
    location=(52.16952247319173, 5.45816115915332),
    zoom_start=19
)

nodesSubset.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 3},
    name='Nodes'
)
jgf.FoliumToPng(m, str( images_path / 'detail_roundabout' ) )
m

In [ ]:
edgesSubset.highway.value_counts(dropna=False)

In [ ]:
transit = gu.load_or_acquire(
    _data_path,
    'nl_transit_ferry',
    osm.get_data_by_custom_criteria,
    logger=logger,
    verbose_args=True,
    custom_filter={
        'route': ['ferry'],
        'public_transport': True
    },
    filter_type="keep"
)     

In [ ]:
transit.explore()